In [1]:
# libary
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy.sparse import coo_matrix
import time
from sklearn.preprocessing import normalize

MIXED-CLUSTER ASSIGNMENT!

In [3]:
# Load the mixed-cluster assignment from CSV
path_cluster = "D:/ITB/Thesis/fcgma/rmfs-sku_to_pod-mayfly-algorithm/Clustering/mixed_cluster_results.csv"
df_mixed = pd.read_csv(path_cluster, sep=';', decimal=',', encoding='utf-8-sig', engine = "python")
df_mixed.info()

<class 'pandas.DataFrame'>
RangeIndex: 5635 entries, 0 to 5634
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   title               5635 non-null   str  
 1   category            5635 non-null   str  
 2   capacity_unit       5635 non-null   str  
 3   item_code           5635 non-null   int64
 4   cluster_local       5635 non-null   int64
 5   cluster_global      5635 non-null   int64
 6   mixed_cluster       5635 non-null   int64
 7   cluster_size        5635 non-null   int64
 8   product_type        5635 non-null   int64
 9   is_new_association  5635 non-null   bool 
 10  pair_id             548 non-null    str  
 11  allocation_type     5635 non-null   str  
dtypes: bool(1), int64(6), str(5)
memory usage: 489.9 KB


In [4]:
# Create a same-cluster indicator matrix from the mixed-cluster assignment
# Pairwise same-cluster indicator matrix
u = df_mixed[['item_code', 'mixed_cluster']].set_index('item_code')
c = u['mixed_cluster'].to_numpy() # Get the cluster labels as a numpy array
M = (c[:, None] == c[None, :]).astype(int) 

# Same product must be 0
np.fill_diagonal(M, 0)

# Final indicator matrix
same_cluster_matrix = pd.DataFrame(M, index = u.index, columns = u.index)
same_cluster_matrix.rename_axis(index=None, inplace=True)
same_cluster_matrix.head()

item_code,10000001001,10000002001,10000003001,10000004001,10000005001,10000007001,10000008001,10000010001,10000011001,10000012001,...,65031238001,65033002001,65033033001,65033086001,65034008001,65103005002,65103009001,65106030001,65106036001,65106084001
10000001001,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10000002001,0,0,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
10000003001,0,0,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10000004001,0,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
10000005001,0,0,1,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
M

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 1, 1],
       [0, 0, 0, ..., 1, 0, 1],
       [0, 0, 0, ..., 1, 1, 0]], shape=(5635, 5635))

In [7]:
# Save same-cluster indicator matrix to CSV
same_cluster_matrix.to_csv("D:/ITB/Thesis/fcgma/rmfs-sku_to_pod-mayfly-algorithm/Clustering/same_cluster_matrix.csv", encoding='utf-8-sig', sep=';', decimal=',')

ASSOCIATION MATRIX

In [8]:
# Create association matrix from the order data
# Load the order data
path_order = "D:/ITB/Thesis/Preprocessing/訂單資料_final.csv"
df_order = pd.read_csv(path_order, encoding='utf-8-sig', sep=';', decimal=',', engine="python")
df_order.head()

,订单号,商品编码,商品名称,商品数量,创建时间,_item_code_key,price_per_piece,sales
0,DO_0000263618,10000005001,可口可樂330ml,6,01/08/2021 00:57,10000005001,13.833333,83.0
1,DO_0000263618,10002014001,維大力汽水Can330ml,6,01/08/2021 00:57,10002014001,13.666667,82.0
2,DO_0000263618,10002018001,黑松沙士330ml,24,01/08/2021 00:57,10002018001,13.833333,332.0
3,DO_0000263618,10014037001,好聖地100椰子水 350ml,3,01/08/2021 00:57,10014037001,29.666667,89.0
4,DO_0000263618,10020007001,伯朗咖啡原味Can240ml:金車伯朗咖啡,24,01/08/2021 00:57,10020007001,20.916667,502.0


In [9]:
df_order['商品编码'].unique()

array([10000005001, 10002014001, 10002018001, ..., 12120043001,
       11110032001, 11010052001], shape=(4784,))

In [10]:
# clean the order data if item code missing from df_mixed
valid_item_codes = set(df_mixed['item_code'])
df_order = df_order[df_order['商品编码'].isin(valid_item_codes)].copy()
df_order['商品编码'].unique()

array([10000005001, 10002014001, 10002018001, ..., 12120043001,
       11110032001, 11010052001], shape=(4748,))

In [11]:
# Convert the '创建时间' column to datetime format
df_order['创建时间'] = pd.to_datetime(df_order['创建时间'], format='%d/%m/%Y %H:%M')
df_order.info()

<class 'pandas.DataFrame'>
Index: 250540 entries, 0 to 250924
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   订单号              250540 non-null  str           
 1   商品编码             250540 non-null  int64         
 2   商品名称             250540 non-null  str           
 3   商品数量             250540 non-null  int64         
 4   创建时间             250540 non-null  datetime64[us]
 5   _item_code_key   250540 non-null  int64         
 6   price_per_piece  250540 non-null  float64       
 7   sales            250540 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(3), str(2)
memory usage: 17.2 MB


In [12]:
# Keep unique product appearances per order
pairs = df_order[['订单号', '商品编码']].dropna().drop_duplicates()

# Convert order and product to integer codes
order_codes = pairs['订单号'].astype('category').cat.codes
product_cat = pairs['商品编码'].astype('category')
product_codes = product_cat.cat.codes

n_orders = order_codes.nunique()
n_products = product_codes.nunique()

# Sparse order-product incidence matrix
X_sparse = coo_matrix(
    (np.ones(len(pairs), dtype=np.int32), (order_codes, product_codes)),
    shape=(n_orders, n_products),
    dtype=np.int32
).tocsr()

# Product-product co-occurrence
association_sparse = (X_sparse.T @ X_sparse).astype(np.int32)

# Remove diagonal
association_sparse.setdiag(0)
association_sparse.eliminate_zeros()


# Convert to dense DataFrame only if product count is small enough
association_matrix = pd.DataFrame(
    association_sparse.toarray(),
    index=product_cat.cat.categories,
    columns=product_cat.cat.categories
)

In [ ]:
# Save association matrix to CSV
association_matrix.rename_axis(columns=None, inplace=True)

In [ ]:
# Normalize the association matrix by 

A = association_matrix.to_numpy(dtype=float)

min_val = A.min()
max_val = A.max()

if max_val > min_val:
    association_matrix_normalized = (A - min_val) / (max_val - min_val)
else:
    association_matrix_normalized = np.zeros_like(A)

association_matrix_normalized = pd.DataFrame(
    association_matrix_normalized,
    index=association_matrix.index,
    columns=association_matrix.columns
)

association_matrix_normalized
association_matrix_normalized.to_csv("D:/ITB/Thesis/Mayfly-Optimization-Algorithm-Python-main/association_matrix_normalized.csv", encoding='utf-8-sig', sep = ';')

In [13]:
unique_value = np.unique(A)
unique_value

NameError: name 'A' is not defined

In [ ]:
# Keep only upper triangle (exclude diagonal) so each pair appears once
arr = association_matrix.to_numpy()
mask = np.triu(np.ones(arr.shape, dtype=bool), k=1)

pairs = (
    association_matrix.where(mask)
    .stack()
    .reset_index()
)
pairs.columns = ['product_a', 'product_b', 'co_occurrence']

#Top pairs (for sanity check)
top_pairs = pairs.sort_values('co_occurrence', ascending=False).head(20)
print("Top 20 co-occurrence pairs:")
display(top_pairs)

Top 20 co-occurrence pairs:


,product_a,product_b,co_occurrence
628913,10021025001,10021026001,567.0
8169082,14010142001,14014014001,452.0
8169079,14010142001,14014006001,447.0
8370685,14014006001,14014014001,442.0
8380989,14014014001,14023056001,332.0
8281839,14011029001,14014005001,321.0
8370738,14014006001,14023056001,309.0
8380937,14014014001,14014015001,303.0
9037194,14112007001,14112013001,289.0
8541588,14023028001,14023056001,287.0


Calculate the number of Pod

In [14]:
# Load library
import numpy as np
import pandas as pd
from fitter import Fitter, get_common_distributions, get_distributions

In [16]:
# Load the data set
df_order = pd.read_csv("D:/ITB/Thesis/Preprocessing/訂單資料_final.csv", encoding='utf-8-sig', sep=';', decimal=',', engine = "python")
df_mixed = pd.read_csv("D:/ITB/Thesis/fcgma/rmfs-sku_to_pod-mayfly-algorithm/Clustering/mixed_cluster_results.csv", sep=';', decimal=',', encoding='utf-8-sig', engine = "python")

In [17]:
# Clean the order data if item code missing from df_mixed
valid_item_codes = set(df_mixed['item_code'])
df_order = df_order[df_order['商品编码'].isin(valid_item_codes)].copy()
df_order.head()

,订单号,商品编码,商品名称,商品数量,创建时间,_item_code_key,price_per_piece,sales
0,DO_0000263618,10000005001,可口可樂330ml,6,01/08/2021 00:57,10000005001,13.833333,83.0
1,DO_0000263618,10002014001,維大力汽水Can330ml,6,01/08/2021 00:57,10002014001,13.666667,82.0
2,DO_0000263618,10002018001,黑松沙士330ml,24,01/08/2021 00:57,10002018001,13.833333,332.0
3,DO_0000263618,10014037001,好聖地100椰子水 350ml,3,01/08/2021 00:57,10014037001,29.666667,89.0
4,DO_0000263618,10020007001,伯朗咖啡原味Can240ml:金車伯朗咖啡,24,01/08/2021 00:57,10020007001,20.916667,502.0


In [18]:
# Identify the distribution type of each SKU

df_indexed = df_order.set_index('商品编码')

# Convert '创建时间' to datetime and group by it
df_order['创建时间'] = pd.to_datetime(df_order['创建时间'], format='%d/%m/%Y %H:%M')

In [27]:
# Calculate mean, std, CV, and minimum inventory for all items
from scipy import stats

minimum_inventory_list = []

# Get all unique item codes
unique_items = df_indexed.index.unique()

nrmse_threshold = 0.2
candidate_distributions = ['norm', 'chi2', 'poisson', 'expon']

for item_code in unique_items:
    try:
        # Get data for this item
        item_data = df_indexed.loc[item_code].copy()

        # Handle single-row vs multi-row selection
        if isinstance(item_data, pd.Series):
            item_data = item_data.to_frame().T

        # Build demand series per timestamp
        item_data['创建时间'] = pd.to_datetime(item_data['创建时间'], format='%d/%m/%Y %H:%M')
        demand = item_data.groupby('创建时间')['商品数量'].sum().to_numpy(dtype=float)
        demand = demand[np.isfinite(demand)]

        # Skip if insufficient data
        if len(demand) < 3:
            continue

        # Empirical stats (used directly for fallback and as stability baseline)
        empirical_mean = float(np.mean(demand))
        empirical_std = float(np.std(demand, ddof=0))
        empirical_cv = (empirical_std / empirical_mean) if empirical_mean > 0 else 0.0

        # Fit distribution
        fitter = Fitter(demand, distributions=candidate_distributions)
        fitter.fit()
        summary_df = fitter.summary(method='sumsquare_error', plot=False)

        # Empirical CDF for NRMSE check
        values, counts = np.unique(demand, return_counts=True)
        emp_pmf = counts / counts.sum()
        emp_cdf = np.cumsum(emp_pmf)

        best_nrmse = np.inf
        dist_type = None
        dist_params = None

        for cand in candidate_distributions:
            params = fitter.fitted_param.get(cand)
            if params is None:
                continue
            try:
                dist = getattr(stats, cand)
                if isinstance(params, dict):
                    model_cdf = dist.cdf(values, **params)
                else:
                    model_cdf = dist.cdf(values, *params)
                nrmse = np.sqrt(np.mean((model_cdf - emp_cdf) ** 2))
            except Exception:
                continue
            if nrmse < best_nrmse:
                best_nrmse = nrmse
                dist_type = cand
                dist_params = params

        if dist_type is None:
            dist_type = 'empirical'
            dist_params = None
            best_nrmse = np.inf

        if not summary_df.empty and dist_type in summary_df.index:
            best_sse = float(summary_df.loc[dist_type, 'sumsquare_error'])
        else:
            best_sse = np.inf

        # Check whether fitted result is numerically stable
        if dist_params is None:
            params_are_finite = False
        else:
            params_iter = dist_params.values() if isinstance(dist_params, dict) else dist_params
            params_are_finite = all(np.isfinite(v) for v in params_iter)
        use_fitted_distribution = bool(best_nrmse < nrmse_threshold and params_are_finite)

        if use_fitted_distribution:
            dist = getattr(stats, dist_type)
            try:
                if isinstance(dist_params, dict):
                    mean = float(dist.mean(**dist_params))
                    std_dev = float(dist.std(**dist_params))
                else:
                    mean = float(dist.mean(*dist_params))
                    std_dev = float(dist.std(*dist_params))
            except Exception:
                use_fitted_distribution = False
                mean = empirical_mean
                std_dev = empirical_std
        else:
            # Poor fit (high NRMSE) or unstable params: use empirical summary stats
            mean = empirical_mean
            std_dev = empirical_std

        # Additional stability checks after parameter-to-moment conversion
        if not np.isfinite(mean) or not np.isfinite(std_dev) or mean <= 0 or std_dev < 0:
            use_fitted_distribution = False
            mean = empirical_mean
            std_dev = empirical_std

        # Service level policy based on CV
        cv = (std_dev / mean) if mean != 0 else 0.0
        if cv <= 0.5:
            service_level = 0.99
        elif cv <= 1.0:
            service_level = 0.95
        else:
            service_level = 0.90

        # Calculate minimum inventory
        if use_fitted_distribution:
            dist = getattr(stats, dist_type)
            try:
                if isinstance(dist_params, dict):
                    minimum_inventory = dist.ppf(service_level, **dist_params)
                else:
                    minimum_inventory = dist.ppf(service_level, *dist_params)
            except Exception:
                minimum_inventory = np.quantile(demand, service_level)
                dist_type = 'empirical'
                use_fitted_distribution = False
        else:
            minimum_inventory = np.quantile(demand, service_level)
            dist_type = 'empirical'

        # If fitted quantile is unstable, fallback to empirical quantile
        if not np.isfinite(minimum_inventory):
            minimum_inventory = np.quantile(demand, service_level)
            dist_type = 'empirical'
            use_fitted_distribution = False

        # Inventory should be non-negative integer units
        minimum_inventory = int(np.ceil(max(0, minimum_inventory)))

        # Append to list
        minimum_inventory_list.append({
            'item_code': item_code,
            'dist_type': dist_type,
            'best_sse': best_sse,
            'best_nrmse': best_nrmse,
            'use_fitted_distribution': use_fitted_distribution,
            'mean': mean,
            'standard_deviation': std_dev,
            'service_level': service_level,
            'cv': cv,
            'minimum_inventory': minimum_inventory,
        })

    except Exception as e:
        print(f"Error processing item {item_code}: {str(e)}")
        continue

# Create DataFrame
minimum_inventory = pd.DataFrame(minimum_inventory_list)
print(f"Calculated distribution stats for {len(minimum_inventory)} items")
display(minimum_inventory.head(10))

c:\Users\lukma\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\scipy\stats\_distn_infrastructure.py:2159: RuntimeWarning: invalid value encountered in divide
  x = np.asarray((x - loc)/scale, dtype=dtyp)
c:\Users\lukma\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\scipy\stats\_distn_infrastructure.py:2159: RuntimeWarning: invalid value encountered in divide
  x = np.asarray((x - loc)/scale, dtype=dtyp)
c:\Users\lukma\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\scipy\stats\_distn_infrastructure.py:2159: RuntimeWarning: invalid value encountered in divide
  x = np.asarray((x - loc)/scale, dtype=dtyp)
c:\Users\lukma\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\scipy\stats\_distn_infrastructure.py:2159: RuntimeWarning: invalid value encountered in divide
  x = np.asarray((x - loc)/scale, dtype=dtyp)
c:\Users\lukma\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\scipy\stats\_distn_infrastructure.py:2159: RuntimeWarning: invalid valu

Calculated distribution stats for 3950 items


,item_code,dist_type,best_sse,best_nrmse,use_fitted_distribution,mean,standard_deviation,service_level,cv,minimum_inventory
0,10000005001,chi2,0.008079,0.069945,True,31.917783,36.200060,0.90,1.134166,76
1,10002014001,norm,0.323780,0.113933,True,22.292683,18.129943,0.95,0.813269,53
2,10002018001,expon,0.006801,0.076682,True,28.917391,22.917391,0.95,0.792512,75
3,10014037001,empirical,148.465825,0.254172,False,4.411765,2.327807,0.95,0.527636,9
4,10020007001,chi2,0.003154,0.056446,True,32.704663,34.214353,0.90,1.046161,76
5,10020011001,empirical,1.199852,0.235797,False,31.807229,17.517260,0.95,0.550732,70
6,10021026001,chi2,0.019460,0.054295,True,5.044286,8.218707,0.90,1.629310,14
7,10032006001,expon,0.011627,0.083490,True,44.701357,38.701357,0.95,0.865776,122
8,12103026001,empirical,358.380746,0.316890,False,5.200000,1.833030,0.99,0.352506,8
9,14011030001,empirical,0.169407,0.215196,False,5.819277,14.464868,0.90,2.485681,12


In [26]:
# Visualization of distribution types
plt.figure(figsize=(10, 6))
sns.countplot(data=minimum_inventory, x='dist_type', order=minimum_inventory['dist_type'].value_counts().index)
plt.title('Distribution Types Used for Minimum Inventory Calculation')
plt.xlabel('Distribution Type')
plt.ylabel('Count')

KeyError: 'dist_type'

<Figure size 1000x600 with 0 Axes>

In [ ]:
''' 
SSE is a fit error measure
-> Smaller SSE means better fit
-> Larger: worse fit

But 'good' depends on:
1. Sample size (more data can lead to larger SSE even if fit is good)
2. Scale of data (larger values can lead to larger SSE)
3. Candidate distributions (if true distribution is not in candidate set, SSE will be larger)
4. Histogram binning (if using histogram-based SSE, choice of bins can affect SSE)

The best distribution will be picked based on the smallest SSE among the candidates (the best fitting).
It's not a proven true distribution, but an approximation

To avoid a bad theoretical distribution onto the SKU, use empirical quantiles instead of fitted distribution 
from historical lead-time demand (1 day)
    -> Plain term: Keep enough stock so that, based on past data, about (service level) of demand cases would have been covered.
'''

" \nSSE is a fit error measure\n-> Smaller SSE means better fit\n-> Larger: worse fit\n\nBut 'good' depends on:\n1. Sample size (more data can lead to larger SSE even if fit is good)\n2. Scale of data (larger values can lead to larger SSE)\n3. Candidate distributions (if true distribution is not in candidate set, SSE will be larger)\n4. Histogram binning (if using histogram-based SSE, choice of bins can affect SSE)\n\nThe best distribution will be picked based on the smallest SSE among the candidates (the best fitting).\nIt's not a proven true distribution, but an approximation\n\nTo avoid a bad theoretical distribution onto the SKU, use empirical quantiles instead of fitted distribution \nfrom historical lead-time demand (1 day)\n    -> Plain term: Keep enough stock so that, based on past data, about (service level) of demand cases would have been covered.\n"

In [23]:
# Save minimum inventory DataFrame to CSV
minimum_inventory.to_csv("D:/ITB/Thesis/fcgma/rmfs-sku_to_pod-mayfly-algorithm/minimum_inventory.csv", sep=';', encoding='utf-8-sig')

In [ ]:
# TEST
from scipy import stats

minimum_inventory_list_test = []

# Get data for this item
item_data = df_indexed.loc[14201020001].copy()

# Handle if single row vs multiple rows
if isinstance(item_data, pd.Series):
    item_data = item_data.to_frame().T

# Convert '创建时间' to datetime and group by it
item_data['创建时间'] = pd.to_datetime(item_data['创建时间'], format='%d/%m/%Y %H:%M')
demand = item_data.groupby('创建时间')['商品数量'].sum().values

if len(demand) < 3:
    raise ValueError('Not enough demand observations to fit distribution (need at least 3).')

# Fit distribution
fitter = Fitter(demand, distributions=['norm', 'chi2', 'poisson', 'expon', 'lognorm'])
fitter.fit()
summary_df = fitter.summary(method='sumsquare_error', plot=False)
best_sse = summary_df['sumsquare_error'].iloc[0]

# Get best distribution
best_dist = fitter.get_best(method='sumsquare_error')
dist_type = list(best_dist.keys())[0]
dist_params = best_dist[dist_type]

# Calculate mean and standard deviation based on distribution type
if dist_type == 'norm':
    mean = dist_params['loc']
    std_dev = dist_params['scale']

elif dist_type == 'chi2':
    df_chi2 = dist_params['df']
    loc = dist_params['loc']
    scale = dist_params['scale']
    mean = df_chi2 * scale + loc
    std_dev = np.sqrt(2 * df_chi2) * scale

elif dist_type == 'poisson':
    mean = dist_params['mu']
    std_dev = np.sqrt(mean)

elif dist_type == 'expon':
    loc = dist_params['loc']
    scale = dist_params['scale']
    mean = scale + loc
    std_dev = scale

elif dist_type == 'lognorm':
    s = dist_params['s']
    loc = dist_params['loc']
    scale = dist_params['scale']
    mean = np.exp(np.log(scale) + s**2 / 2) + loc
    std_dev = np.sqrt((np.exp(s**2) - 1) * np.exp(2 * np.log(scale) + s**2))

else:
    raise ValueError(f'Unsupported distribution: {dist_type}')

# Calculate Coefficient of Variation
cv = (std_dev / mean) if mean != 0 else 0

# Service level policy based on CV
if cv <= 0.5:
    service_level = 0.99
elif cv <= 1.0:
    service_level = 0.95
else:
    service_level = 0.90

# Calculate minimum inventory from fitted distribution quantile (PPF)
if dist_type == 'norm':
    minimum_inventory = stats.norm.ppf(service_level, loc=dist_params['loc'], scale=dist_params['scale'])
elif dist_type == 'chi2':
    minimum_inventory = stats.chi2.ppf(
        service_level,
        dist_params['df'],
        loc=dist_params['loc'],
        scale=dist_params['scale']
    )
elif dist_type == 'poisson':
    minimum_inventory = stats.poisson.ppf(service_level, mu=dist_params['mu'])
elif dist_type == 'expon':
    minimum_inventory = stats.expon.ppf(service_level, loc=dist_params['loc'], scale=dist_params['scale'])
elif dist_type == 'lognorm':
    minimum_inventory = stats.lognorm.ppf(
        service_level,
        dist_params['s'],
        loc=dist_params['loc'],
        scale=dist_params['scale']
    )

# Inventory should be non-negative integer units
minimum_inventory = int(np.ceil(max(0, minimum_inventory)))

# Append to list
minimum_inventory_list_test.append({
    'item_code': 14201020001,
    'dist_type': dist_type,
    'best_sse': best_sse,
    'mean': mean,
    'standard_deviation': std_dev,
    'cv': cv,
    'service_level': service_level,
    'minimum_inventory': minimum_inventory
})

# Create DataFrame
minimum_inventory_test = pd.DataFrame(minimum_inventory_list_test)
print(f"Calculated distribution stats for {len(minimum_inventory_test)} items")
display(minimum_inventory_test.head(10))
print(summary_df)

Calculated distribution stats for 1 items


,item_code,dist_type,best_sse,mean,standard_deviation,cv,service_level,minimum_inventory
0,14201020001,lognorm,159.473152,2.350358e+56,1.940085e+117,8.254426e+60,0.9,59773


         sumsquare_error         aic         bic     kl_div  ks_statistic  \
lognorm       159.473152 -118.584810 -117.677055  14.706060      0.433982   
chi2          159.547275  -15.685055  -14.777300  20.103169      0.300000   
expon         165.346512   29.247285   29.852455  20.588411      0.300000   
norm          168.529371   35.814668   36.419838  20.812970      0.199729   
poisson              inf         inf         inf        inf           inf   

         ks_pvalue  
lognorm   0.031437  
chi2      0.270536  
expon     0.270536  
norm      0.750111  
poisson   0.000000  
